95 mins 44.3s

## Load libraries

In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm
from collections import defaultdict
from sklearn.base import BaseEstimator, TransformerMixin

## Config

In [2]:

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# ---- lag plan (always 1 & 12; variants with 2–6; optional 24) ----
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

spatial_cols = ["area_km2", "centroid_x", "centroid_y", "CoL_distance_km"]

# everything continuous except the spatial ones goes into "other continuous"
other_continuous_cols = [c for c in continuous_cols if c not in spatial_cols]
other_cols = other_continuous_cols + categorical_cols

# ---- block-wise kernel grid ----
svm_param_grid = {
    "alpha": [1e-6, 1e-5, 1e-4],
    "epsilon": [0.01, 0.1],

    # temporal block: multi-scale RBF
    "temporal_gammas": [(0.05, 0.1), (0.05, 0.1, 0.2)],
    "temporal_n_components": [150, 300],   # per gamma

    # spatial block: single RBF
    "spatial_gamma": [0.01, 0.05],         # usually smoother than temporal
    "spatial_n_components": [150, 300],
}

# split grid into (kernel map params) and (svr params)
kernel_grid = {
    "temporal_gammas": svm_param_grid["temporal_gammas"],
    "temporal_n_components": svm_param_grid["temporal_n_components"],
    "spatial_gamma": svm_param_grid["spatial_gamma"],
    "spatial_n_components": svm_param_grid["spatial_n_components"],
}
svr_grid = {
    "alpha": svm_param_grid["alpha"],
    "epsilon": svm_param_grid["epsilon"],
}


## Evaluation metric functions

In [3]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale


## Load data

In [4]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

mask_tv = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask_tv].copy()

# all unique months in train+val window
dates = sorted(df_tv[TIME_COL].unique())

## multi kernel

In [5]:
class MultiRBFFeatures(BaseEstimator, TransformerMixin):
    """
    Concatenate multiple RBFSampler feature maps (multi-scale RBF).
    """
    def __init__(self, gammas=(0.05, 0.1, 0.2), n_components=200, random_state=42):
        self.gammas = tuple(gammas)
        self.n_components = int(n_components)
        self.random_state = int(random_state)

    def fit(self, X, y=None):
        self.samplers_ = []
        for i, g in enumerate(self.gammas):
            s = RBFSampler(
                gamma=float(g),
                n_components=self.n_components,
                random_state=self.random_state + i
            )
            s.fit(X)
            self.samplers_.append(s)
        return self

    def transform(self, X):
        Zs = [s.transform(X) for s in self.samplers_]
        return np.hstack(Zs)

## Rolling STL feature builder

In [6]:
def add_rolling_stl_components(
    df: pd.DataFrame,
    entity_col: str,
    time_col: str,
    target_col: str,
    period: int = 12,
    min_history: int = 24,
    robust: bool = True,
    show_progress: bool = True,
) -> pd.DataFrame:
    """
    Time-safe rolling STL (one-sided).
    For each entity and each time t, fit STL on y[:t] and assign the last component values to time t.

    Outputs columns:
      - stl_trend
      - stl_seasonal
      - stl_resid

    Notes:
    - This is computationally heavier than "fit once on train then extrapolate".
    - It avoids leakage because STL at time t uses only <= t data.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    grouped = df.groupby(entity_col, sort=False)
    iterator = grouped if not show_progress else tqdm(grouped, desc="Rolling STL by LA", leave=False)

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        idx = sub.index.values

        # Rolling one-sided STL: start only when we have enough history
        for t in range(min_history - 1, len(y)):
            y_hist = y[: t + 1]

            # Skip if history contains NaNs
            if np.isnan(y_hist).any():
                continue

            try:
                stl = STL(y_hist, period=period, robust=robust)
                res = stl.fit()

                df.loc[idx[t], "stl_trend"] = float(res.trend[-1])
                df.loc[idx[t], "stl_seasonal"] = float(res.seasonal[-1])
                df.loc[idx[t], "stl_resid"] = float(res.resid[-1])

            except Exception:
                # If STL fails for numeric reasons at this t, leave NaNs
                continue

    return df

## Training with STL + Rolling CV

In [9]:
# =========================================================
# CV STRUCTURE:
#   outer loop   = folds (STL + all lags computed once per fold)
#   mid loop     = lag_set (subselect lag columns, scale, etc.)
#   inner loop   = RF params (fit, predict, metrics)
# =========================================================

# metrics_store[(lag_tuple, params_key)] = dict of metric lists across folds
metrics_store = {}

# Pre-build list of folds based on dates
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > len(dates):
        break

    train_start = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end   = dates[train_end_idx - 1]
    val_start   = dates[val_start_idx]
    val_end     = dates[val_end_idx - 1]

    fold_specs.append((train_start, train_end, val_start, val_end))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

# =========================================================
# MAIN LOOP: FOLDS → (lag_set → params)
# =========================================================
for fold_no, (train_start, train_end, val_start, val_end) in enumerate(fold_specs, start=1):
    print(
        f"\n=== Fold {fold_no}: Train {train_start:%Y-%m}–{train_end:%Y-%m}, "
        f"Val {val_start:%Y-%m}–{val_end:%Y-%m} ==="
    )

    # ---- slice fold train/val ----
    mask_train = (df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)
    mask_val   = (df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)

    fold_train = df_tv.loc[mask_train].copy()
    fold_val   = df_tv.loc[mask_val].copy()

    fold_train["is_train"] = True
    fold_val["is_train"]   = False

    combined = pd.concat([fold_train, fold_val], axis=0).sort_values([ENTITY_COL, TIME_COL])

    # ---- rolling STL (time-safe) ----
    combined = add_rolling_stl_components(
        combined,
        entity_col=ENTITY_COL,
        time_col=TIME_COL,
        target_col=TARGET_COL,
        period=12,
        min_history=24,
        robust=True,
        show_progress=True,
    )
    combined = combined.sort_values([ENTITY_COL, TIME_COL])

    # ---- create lag columns for ALL lags (superset) on STL components ----
    for lag in all_lags:
        for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
            col = f"{comp}_lag{lag}"
            combined[col] = combined.groupby(ENTITY_COL)[comp].shift(lag)

    # ---------------------------------------------------------
    # For this fold: loop over lag_set, then kernel-map params, then SVR params
    # ---------------------------------------------------------
    for lag_set in lag_combinations:
        print(f"  Lag set: {lag_set}")

        lag_cols = [
            f"{comp}_lag{lag}"
            for comp in ["stl_trend", "stl_seasonal", "stl_resid"]
            for lag in lag_set
        ]
        
        best_rmse_lag = np.inf
        best_mae_lag = np.inf
        best_params_lag = None
        best_kernel_lag = None
        best_svr_lag = None

        # split back into train/val
        fold_train_lag = combined[combined["is_train"]].copy()
        fold_val_lag   = combined[~combined["is_train"]].copy()

        # require ALL FEATURES present
        full_feature_cols = continuous_cols + categorical_cols + lag_cols
        fold_train_lag = fold_train_lag.dropna(subset=full_feature_cols)
        fold_val_lag   = fold_val_lag.dropna(subset=full_feature_cols)

        if fold_train_lag.empty or fold_val_lag.empty:
            print("    (skip: no data after feature drop)")
            continue

        # ----------------------
        # Build blocked X, y
        # ----------------------
        y_train_raw = fold_train_lag[TARGET_COL].values.reshape(-1, 1)
        y_val_raw   = fold_val_lag[TARGET_COL].values.reshape(-1, 1)

        # Temporal block (STL lags)
        X_train_temp = fold_train_lag[lag_cols].copy()
        X_val_temp   = fold_val_lag[lag_cols].copy()

        # Spatial block
        X_train_spat = fold_train_lag[spatial_cols].copy()
        X_val_spat   = fold_val_lag[spatial_cols].copy()

        # Other block: remaining continuous + categoricals (linear)
        X_train_other = fold_train_lag[other_cols].copy()
        X_val_other   = fold_val_lag[other_cols].copy()

        # ----------------------
        # Scale each block (train only)
        # ----------------------
        temp_scaler = StandardScaler()
        X_train_temp = temp_scaler.fit_transform(X_train_temp)
        X_val_temp   = temp_scaler.transform(X_val_temp)

        spat_scaler = StandardScaler()
        X_train_spat = spat_scaler.fit_transform(X_train_spat)
        X_val_spat   = spat_scaler.transform(X_val_spat)

        other_scaler = StandardScaler()
        X_train_other_scaled = X_train_other.copy()
        X_val_other_scaled   = X_val_other.copy()

        X_train_other_scaled[other_continuous_cols] = other_scaler.fit_transform(
            X_train_other_scaled[other_continuous_cols]
        )
        X_val_other_scaled[other_continuous_cols] = other_scaler.transform(
            X_val_other_scaled[other_continuous_cols]
        )

        Z_train_other = X_train_other_scaled.values
        Z_val_other   = X_val_other_scaled.values

        # ----------------------
        # Scale y once per (fold, lag_set)
        # ----------------------
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train_raw).ravel()
        y_val_scaled   = y_scaler.transform(y_val_raw).ravel()

        # =====================================================
        # STAGE A: Build/cache Z for each KERNEL configuration
        # =====================================================
        Z_cache = {}  # kernel_key -> (Z_train, Z_val)

        for kparams in ParameterGrid(kernel_grid):
            kkey = tuple(sorted(kparams.items()))
            if kkey not in Z_cache:
                # Temporal: multi-scale RBF features
                temporal_map = MultiRBFFeatures(
                    gammas=kparams["temporal_gammas"],
                    n_components=kparams["temporal_n_components"],
                    random_state=42,
                )
                Z_train_temp = temporal_map.fit_transform(X_train_temp)
                Z_val_temp   = temporal_map.transform(X_val_temp)

                # Spatial: single RBF features
                spatial_map = RBFSampler(
                    gamma=kparams["spatial_gamma"],
                    n_components=kparams["spatial_n_components"],
                    random_state=123,
                )
                Z_train_spat = spatial_map.fit_transform(X_train_spat)
                Z_val_spat   = spatial_map.transform(X_val_spat)

                # Concatenate blocks
                Z_train = np.hstack([Z_train_temp, Z_train_spat, Z_train_other])
                Z_val   = np.hstack([Z_val_temp,   Z_val_spat,   Z_val_other])

                # ---- SCALE Z (fit on train, apply to val) ----
                Z_scaler = StandardScaler()
                Z_train = Z_scaler.fit_transform(Z_train)
                Z_val   = Z_scaler.transform(Z_val)

                Z_cache[kkey] = (Z_train, Z_val)

            # =====================================================
            # STAGE B: For this kernel map, loop over SVR params
            # =====================================================
            Z_train, Z_val = Z_cache[kkey]

            for sparams in ParameterGrid(svr_grid):
                params = {**kparams, **sparams}  # full params for logging

                print(f"    Kernel: {kparams} | SVR: {sparams}")

                lag_key = tuple(lag_set)
                params_key = tuple(sorted(params.items()))
                key = (lag_key, params_key)

                if key not in metrics_store:
                    metrics_store[key] = {
                        "lag_set": lag_key,
                        "params": params,
                        "mae": [],
                        "rmse": [],
                        "smape": [],
                        "mase": [],
                        "folds": 0,
                    }

                
                svr = SGDRegressor(
                    loss="epsilon_insensitive",
                    penalty="l2",
                    epsilon=sparams["epsilon"],
                    alpha=sparams["alpha"],
                    learning_rate="invscaling",
                    eta0=0.01,
                    max_iter=3000,
                    tol=1e-3,
                    random_state=42,
                )
                svr.fit(Z_train, y_train_scaled)

                y_pred_scaled = svr.predict(Z_val)
                y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

                y_val_true   = y_val_raw.ravel()
                y_train_true = y_train_raw.ravel()

                metrics_store[key]["mae"].append(mae(y_val_true, y_pred))
                metrics_store[key]["rmse"].append(rmse(y_val_true, y_pred))
                metrics_store[key]["smape"].append(smape(y_val_true, y_pred))
                metrics_store[key]["mase"].append(mase(y_val_true, y_pred, y_train_true))
                metrics_store[key]["folds"] += 1

                cur_mae  = mae(y_val_true, y_pred)
                cur_rmse = rmse(y_val_true, y_pred)

                # store metrics (your existing code)
                metrics_store[key]["mae"].append(cur_mae)
                metrics_store[key]["rmse"].append(cur_rmse)
                # ... etc

                # update best-for-this-lag-set (this fold)
                if cur_rmse < best_rmse_lag:
                    best_rmse_lag = cur_rmse
                    best_mae_lag = cur_mae
                    best_params_lag = params
                    best_kernel_lag = kparams
                    best_svr_lag = sparams
             # ADD print here (after both loops)
        print(
            f"  >>> BEST for lag set {lag_set} (fold {fold_no}): "
            f"RMSE={best_rmse_lag:,.3f} | MAE={best_mae_lag:,.3f}\n"
            f"      kernel={best_kernel_lag}\n"
            f"      svr={best_svr_lag}"
        )

Number of folds: 5

=== Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03 ===


  Lag set: [1, 12]
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 0.0001, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha'

  Lag set: [1, 12]
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 0.0001, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha'

  Lag set: [1, 12]
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 0.0001, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha'

  Lag set: [1, 12]
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 0.0001, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha'

  Lag set: [1, 12]
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-06, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 1e-05, 'epsilon': 0.1}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha': 0.0001, 'epsilon': 0.01}
    Kernel: {'spatial_gamma': 0.01, 'spatial_n_components': 150, 'temporal_gammas': (0.05, 0.1), 'temporal_n_components': 150} | SVR: {'alpha'

## Results

In [10]:
rows = []
for key, val in metrics_store.items():
    if val["folds"] == 0:
        continue
    rows.append({
        "model_type": "SGD_EPS_INSENSITIVE_BLOCKWISE_RFF",
        "lag_set": val["lag_set"],
        "params": val["params"],
        "folds": val["folds"],
        "MAE_mean":   float(np.mean(val["mae"])),
        "MAE_std":    float(np.std(val["mae"])),
        "RMSE_mean":  float(np.mean(val["rmse"])),
        "RMSE_std":   float(np.std(val["rmse"])),
        "sMAPE_mean": float(np.mean(val["smape"])),
        "sMAPE_std":  float(np.std(val["smape"])),
        "MASE_mean":  float(np.mean(val["mase"])),
        "MASE_std":   float(np.std(val["mase"])),
    })

results_df = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
print(results_df.head(20))
results_df.to_csv("../../results/svm_leakfree_stl_rollingcv_results.csv", index=False)

                           model_type                 lag_set  \
0   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF           (1, 2, 3, 12)   
1   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF             (1, 12, 24)   
2   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF          (1, 2, 12, 24)   
3   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF       (1, 2, 3, 12, 24)   
4   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF          (1, 2, 12, 24)   
5   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF       (1, 2, 3, 12, 24)   
6   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF          (1, 2, 12, 24)   
7   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF  (1, 2, 3, 4, 5, 6, 12)   
8   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF       (1, 2, 3, 12, 24)   
9   SGD_EPS_INSENSITIVE_BLOCKWISE_RFF             (1, 12, 24)   
10  SGD_EPS_INSENSITIVE_BLOCKWISE_RFF  (1, 2, 3, 4, 5, 6, 12)   
11  SGD_EPS_INSENSITIVE_BLOCKWISE_RFF             (1, 12, 24)   
12  SGD_EPS_INSENSITIVE_BLOCKWISE_RFF              (1, 2, 12)   
13  SGD_EPS_INSENSITIVE_BLOCKWISE_RFF             (1, 12, 24)   
14  SGD_EPS_INSENSITIVE_B

In [ ]:
print(results_df.head(20))

      model_type     lag_set  \
0   SparseGP_RFF     (1, 12)   
1   SparseGP_RFF     (1, 12)   
2   SparseGP_RFF     (1, 12)   
3   SparseGP_RFF     (1, 12)   
4   SparseGP_RFF     (1, 12)   
5   SparseGP_RFF     (1, 12)   
6   SparseGP_RFF  (1, 2, 12)   
7   SparseGP_RFF  (1, 2, 12)   
8   SparseGP_RFF  (1, 2, 12)   
9   SparseGP_RFF  (1, 2, 12)   
10  SparseGP_RFF  (1, 2, 12)   
11  SparseGP_RFF  (1, 2, 12)   
12  SparseGP_RFF     (1, 12)   
13  SparseGP_RFF     (1, 12)   
14  SparseGP_RFF     (1, 12)   
15  SparseGP_RFF     (1, 12)   
16  SparseGP_RFF     (1, 12)   
17  SparseGP_RFF     (1, 12)   
18  SparseGP_RFF  (1, 2, 12)   
19  SparseGP_RFF  (1, 2, 12)   

                                               params  folds      MAE_mean  \
0   {'C': 1.0, 'epsilon': 0.01, 'gamma': 0.05, 'n_...      5  53491.396274   
1   {'C': 1.0, 'epsilon': 0.1, 'gamma': 0.05, 'n_c...      5  53491.396274   
2   {'C': 0.1, 'epsilon': 0.1, 'gamma': 0.05, 'n_c...      5  53491.396274   
3   {'C': 0.1, 